# Phase 1 — Dataset Preparation
This notebook loads datasets via **ir_datasets**, performs lightweight cleaning and truncation, and exports:
- `corpus.jsonl` — document id, cleaned text, length (tokens)
- `queries.jsonl` — query id, cleaned text
- `qrels.jsonl` — qrels triples

> Matches SOP Step 1 (Dataset Setup & Preprocessing).

In [1]:
import sys, os, subprocess
os.environ["PIP_NO_CACHE_DIR"]="1"
subprocess.run([sys.executable, "-m", "pip", "install", "-U", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install",
                "--index-url","https://download.pytorch.org/whl/cu121",
                "torch==2.4.1"], check=True)

# verify cuDNN wheel is present
import pathlib, importlib
import nvidia.cudnn as nc
libdir = pathlib.Path(nc.__file__).parent / "lib"
print("cuDNN libs:", [p.name for p in libdir.glob("libcudnn*.so*")])


Looking in indexes: https://download.pytorch.org/whl/cu121
cuDNN libs: ['libcudnn_engines_runtime_compiled.so.9', 'libcudnn_cnn.so.9', 'libcudnn_heuristic.so.9', 'libcudnn_adv.so.9', 'libcudnn_graph.so.9', 'libcudnn_ops.so.9', 'libcudnn.so.9', 'libcudnn_engines_precompiled.so.9']


In [2]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "fastai"], check=False)
# (optional) also remove fastai extras you don't need:
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y",
                "fastcore","fastdownload","fastprogress"], check=False)

# sanity check for remaining conflicts
subprocess.run([sys.executable, "-m", "pip", "check"])


No broken requirements found.


CompletedProcess(args=['/home/mallarapuhemavarshini/anaconda3/bin/python', '-m', 'pip', 'check'], returncode=0)

In [3]:
!pip install -r requirements.txt

In [8]:
import os
os.environ.pop('IR_DATASETS_HOME', None)

'/path/with/space/ir_datasets'

In [9]:
# shell cell
!python3 -m ir_datasets list | grep trec-covid


beir/trec-covid
cord19/fulltext/trec-covid
cord19/trec-covid
cord19/trec-covid/round1
cord19/trec-covid/round2
cord19/trec-covid/round3
cord19/trec-covid/round4
cord19/trec-covid/round5


In [10]:
from __future__ import annotations
import os, re, json
from pathlib import Path
from typing import Iterable, Dict
import ir_datasets
from tqdm import tqdm

WORK_DIR = Path("./work")
#DATASET = "beir/trec-covid/test"   # ← dev → test
DATASET = "beir/trec-covid"
#ds = ir_datasets.load(DATASET)

WORK_DIR.mkdir(parents=True, exist_ok=True)

try:
    ds = ir_datasets.load(DATASET)
except KeyError:
    raise SystemExit(
        f"Dataset key '{DATASET}' not found. "
        "Try installing BEIR extras: pip install -U 'ir_datasets[beir]' "
        "or list available names with: python -m ir_datasets list"
    )

ds_dir = WORK_DIR / "datasets" / DATASET.replace("/", "_")
ds_dir.mkdir(parents=True, exist_ok=True)


In [11]:
from __future__ import annotations
import os, re, json
from pathlib import Path
from typing import Iterable, Dict
import ir_datasets
from tqdm import tqdm

WORK_DIR = Path("./work")               # change if needed
DATASET = "beir/trec-covid"
# e.g., 'msmarco-passage/train', 'beir/nfcorpus/test'
#DATASET = "msmarco-passage/train"
WORK_DIR.mkdir(parents=True, exist_ok=True)
ds = ir_datasets.load(DATASET)
ds_dir = WORK_DIR / "datasets" / DATASET.replace("/", "_")
ds_dir.mkdir(parents=True, exist_ok=True)

def clean_text(text: str) -> str:
    if text is None: return ""
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^a-z0-9\s\-\.,;:!?/()'\"]+", " ", text)
    return text.strip()

def tokenize_simple(text: str):
    import re
    return re.findall(r"[a-z0-9]+", text.lower())

def save_jsonl(path: Path, rows: Iterable[dict]):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

In [12]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected — running on CPU.")


PyTorch version: 2.4.1+cu121
CUDA available: True
GPU name: NVIDIA GeForce RTX 3050 Laptop GPU


In [14]:
# Export corpus (clean + truncate for transformer inputs)
MAX_TOK = 512
MAX_DOCS = 170000   # ← small test limit; BEIR/TREC-COVID has only ~170k docs anyway

corpus_out = ds_dir / "corpus.jsonl"

# Always overwrite for a fresh start (recommended)
with corpus_out.open("w", encoding="utf-8") as f:
    for i, d in enumerate(tqdm(ds.docs_iter(), desc=f"Export corpus (up to {MAX_DOCS})")):
        if i >= MAX_DOCS:
            break
        txt = ""
        if hasattr(d, "text") and d.text:
            txt = d.text
        if hasattr(d, "title") and d.title:
            txt = (d.title + " " + txt).strip()
        txt = clean_text(txt)
        toks = tokenize_simple(txt)[:MAX_TOK]
        txt = " ".join(toks)
        f.write(json.dumps(
            {"doc_id": str(d.doc_id), "text": txt, "len": len(toks)},
            ensure_ascii=False
        ) + "\n")

print(f"✅ Exported {min(i + 1, MAX_DOCS)} documents to {corpus_out}")


Export corpus (up to 170000): 170000it [00:13, 12409.64it/s]

✅ Exported 170000 documents to work/datasets/beir_trec-covid/corpus.jsonl


In [15]:
# Export queries
queries_out = ds_dir / "queries.jsonl"
if not queries_out.exists():
    with queries_out.open("w", encoding="utf-8") as f:
        for q in tqdm(ds.queries_iter(), desc="Export queries"):
            txt = clean_text(getattr(q, "text", ""))
            f.write(json.dumps({"qid": str(q.query_id), "text": txt}, ensure_ascii=False) + "\n")
else:
    print("queries.jsonl already exists")

Export queries: 50it [00:00, 5876.02it/s]


In [16]:
# Export qrels
qrels_out = ds_dir / "qrels.jsonl"
if not qrels_out.exists():
    with qrels_out.open("w", encoding="utf-8") as f:
        for r in tqdm(ds.qrels_iter(), desc="Export qrels"):
            f.write(json.dumps({"qid": str(r.query_id), "doc_id": str(r.doc_id), "rel": int(r.relevance)}) + "\n")
else:
    print("qrels.jsonl already exists")

Export qrels: 66336it [00:00, 332607.33it/s]


✅ **Done.** You can now generate stratified subsets from these JSONL files in Notebook 2.